# Funciones avanzadas y análisis de datos en R

![](src/img/logo_utb.png){width=40%}
![](src/img/logo_etd.png){width=40%}
- **Profesor:**
- **Fernando Salcedo Mejía, Eco Msc.**
- Programa de Ciencias de Datos | Escuela de transformación digital.
- 2026-1

## 1. Funciones avanzadas: lapply, sapply, Reduce

In [ ]:
# Función para calcular el cuadrado
cuadrado <- function(x) x^2
cuadrado(4)

## lapply

- Aplica una función a una lista o vector de forma iterativa.
- Siempre devuelve una lista
- Variantes :
    - sapply() : aplicar función y simplificar resultado. Usualmente un vector
    - tapply() : aplicar función a subconjuntos de datos

In [ ]:
# Vector de ejemplo
numeros <- c(1, 2, 3, 4, 5)

# lapply: aplicar función a cada elemento
print(lapply(numeros, function(x) x^2))
print(lapply(numeros, cuadrado))

In [ ]:
# sapply: aplicar función y simplificar resultado
print(sapply(numeros, cuadrado))
# tapply: aplicar función a subconjuntos de datos
grupos <- c("A", "A", "B", "B", "C")
print(tapply(numeros, grupos, sum))

## Filter

-  Filtra elementos que cumplen una condición lógica.

In [ ]:
# Filter: filtrar números pares
Filter(function(x) x %% 2 == 0, numeros)

## reduce 
Reduce una secuencia a un único valor acumulado.

In [ ]:
# Reduce: sumar y multiplicar
print(Reduce(`+`, numeros))
print(Reduce(`*`, numeros))

## 2. Uso de funciones sobre data.frames

- En data.frame podemos usar:
    - apply() que opera por filas (1) o columnas (2)
    - lappy() que opera por columnas

In [ ]:
datos <- data.frame(
  A = c(10, 20, 30, 40),
  B = c(5, 7, 9, 11),
  C = c('x', 'y', 'z', 'w')
)

# Media por columna numérica
print(apply(datos[c('A','B')], 2, mean))

# Media por columna numérica con lapply
print(lapply(datos[c('A','B')], mean))

# Estandarización (z-score)
print(lapply(datos[c('A','B')], function(x) (x - mean(x)) / sd(x)))

## 3. Datos Palmer Penguins

In [ ]:
library(dplyr)
library(readr)

datos_github <- 'https://raw.githubusercontent.com/fersalme/programacion-python-r/refs/heads/main/datos/palmerpenguins_extended.csv'
df_pinguinos <- read_csv(datos_github, show_col_types = FALSE)

head(df_pinguinos)

In [ ]:
# Clasificar pingüinos por tamaño del pico
df_pinguinos <- df_pinguinos |>
  mutate(tamano = ifelse(bill_length_mm > 40, 'grande', 'pequeño'))

df_pinguinos |>
select(bill_length_mm, tamano) |> 
head()

In [ ]:
# Aplicar mutiples condiciones con case_when
df_pinguinos <- df_pinguinos |>
  mutate(
    tamano = case_when(
    bill_length_mm < 35 ~ 'pequeño',
    bill_length_mm >= 35 & bill_length_mm <= 45 ~ 'mediano',
    bill_length_mm > 45 ~ 'grande',
    .default = NA_character_
  )
)

df_pinguinos |>
select(bill_length_mm, tamano) |> 
head()

## 4. Agrupamientos y estadísticas descriptivas

In [ ]:
# Media de longitud del pico por especie
df_pinguinos |>
  group_by(species)|>
  summarise(media_bill_length = mean(bill_length_mm, na.rm = TRUE))

In [ ]:
# Estadísticas descriptivas
df_pinguinos |> 
  group_by(species) |> 
  summarise(
    n_pinguinos = n(),
    mean_bill_length = mean(bill_length_mm, na.rm = TRUE),
    median_bill_length = median(bill_length_mm, na.rm = TRUE),
    sd_bill_length = sd(bill_length_mm, na.rm = TRUE)
  )

## 5. Tablas dinámicas (pivot tables)

pivot_wider() es una función del paquete tidyr (ecosistema tidyverse) que sirve para pasar datos de formato largo a formato ancho.

![](https://r4ds.hadley.nz/diagrams/tidy-data/variables.png)

In [ ]:
library(tidyr)

# Tabla dinámica equivalente a pivot_table
df_pinguinos |> 
  group_by(species, sex) |> 
  summarise(media_peso = mean(body_mass_g, na.rm = TRUE)) |> 
  pivot_wider(names_from = sex, values_from = media_peso)

## Tarea 1.
- Usando mutate() y case_when() crea una categoria de peso para los pinguinos según el cuartil de peso 25% bajo peso, 50% peso medio, 75% pesado

In [ ]:
# TU CODIGO AQUÍ

## Tarea 2.

- Crear una tabla reporte del total de pinguinos por especie y métrica de salud (health_metrics)
- Crear una tabla con la proporción de pinguinos por métrica de salud según especie

In [ ]:
# TU CODIGO AQUÍ

## 6. Combinación y fusión de datasets en dplyr
- Un join sirve para combinar dos tablas (data.frame / tibble) usando una o más columnas clave.

| Join en dplyr     | Qué hace                                                                 | Filas que conserva                   | Uso típico |
|------------------|--------------------------------------------------------------------------|--------------------------------------|-----------|
| `inner_join()`   | Devuelve solo las filas que tienen coincidencia en ambas tablas          | Solo coincidencias                   | Análisis con datos completos |
| `left_join()`    | Mantiene todas las filas de la tabla izquierda                            | Todas las de la izquierda            | Enriquecer una tabla principal |
| `right_join()`   | Mantiene todas las filas de la tabla derecha                              | Todas las de la derecha              | Poco usado (mejor invertir y usar `left_join`) |
| `full_join()`    | Mantiene todas las filas de ambas tablas                                  | Todas las de ambas                   | Auditoría y control de datos |
| `semi_join()`    | Filtra la tabla izquierda si hay coincidencia (no añade columnas)        | Coincidencias de la izquierda        | Filtrar usando otra tabla |
| `anti_join()`    | Filtra la tabla izquierda sin coincidencia                                | No coincidencias de la izquierda     | Detectar faltantes o errores |


![](https://r4ds.hadley.nz/diagrams/join/inner.png)
![](https://r4ds.hadley.nz/diagrams/join/left.png)



In [ ]:
df1 <- data.frame(id = c(1,2,3), nombre = c('Ana','Luis','María'))
df2 <- data.frame(id = c(1,2,4), nota = c(4.5,3.8,4.9))

print("DataFrame 1:")
print(df1)
print("DataFrame 2:")
print(df2)

In [ ]:
# cruzar los DataFrames usando merge solo lo que coincide en ambos DataFrames
print("Merge (inner_join) por 'id':")
df_merge <- inner_join(df1, df2, by = 'id')
print(df_merge)

In [ ]:
# cruzar los DataFrames usando merge con un left join para mantener todos los registros del primer DataFrame
print("Merge (left_join) por 'id':")
df_merge_left = left_join(df1, df2, by = 'id')
print(df_merge_left)


In [ ]:
# concatenar filas
df_q1 <- data.frame(mes=c('Ene','Feb','Mar'), ventas=c(1200000,1500000,1100000))
df_q2 <- data.frame(mes=c('Abr','May','Jun'), ventas=c(1800000,2100000,1900000))

print("DataFrame Q1:")
print(df_q1)
print("DataFrame Q2:")
print(df_q2)

In [ ]:
# Pegar los DataFrames usando concat para apilar filas
print("Concatenar DataFrames (apilar filas):")
df_ventas_semestre = bind_rows(df_q1, df_q2)
print(df_ventas_semestre)

## Tarea 3:

- Usando los datos de COVID-19 de NUEVOS casos confirmados (time_series_covid19_confirmed_global), NUEVAS muertes (time_series_covid19_deaths_global) y NUEVOS recuperados (time_series_covid19_recovered_global) crea un dataframen único con esta estructura:

| country_region | fecha      | casos | fallecidos | recuperados |
|----------------|------------|-------|------------|-------------|
| Colombia       | 2020-03-04 | 1     | 0          | 0           |

- Datos :
    - time_series_covid19_confirmed_global : https://raw.githubusercontent.com/fersalme/programacion-python-r/refs/heads/main/datos/time_series_covid19_confirmed_global.csv
    - time_series_covid19_recovered_global : https://raw.githubusercontent.com/fersalme/programacion-python-r/refs/heads/main/datos/time_series_covid19_recovered_global.csv
    - time_series_covid19_deaths_global : https://raw.githubusercontent.com/fersalme/programacion-python-r/refs/heads/main/datos/time_series_covid19_deaths_global.csv

- Reporta en una tabla resumen el total de casos, muertes y recuperados para Colombia.

In [ ]:
# TU CODIGO AQUÍ